# Krish Naik Agentic AI 3.0 Course Langchain-Tools

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

# Tools

###  Tools are just methods with proper defined input and output and description

In [2]:
from langchain.chat_models import init_chat_model
from pydantic import BaseModel
from langchain_core.tools import tool

In [3]:
model = init_chat_model('openai:gpt-5-mini')

In [5]:
class MovieShows(BaseModel):
  name : str
  timing:str

response = model.with_structured_output(MovieShows).invoke("Is Interstellar showing tonight at 7pm at the Downtown cinema ?")

In [6]:
response

MovieShows(name='Interstellar', timing='I don’t have access to live showtime data. Would you like me to look up tonight’s 7:00 PM showing at Downtown Cinema (please confirm the city or provide the cinema address)?')

#### sample tool method with decorator, arguments, return type, doctype

In [9]:
@tool
def check_showtimes(movie_title:str) -> str:
  """Check available showtimes for a movie at the cinema.

  Args:
      movie_title: The exact title of the movie to check
  """
  fake_showtimes = {
      "interstellar": "7:00 PM and 10:15 PM",
      "dune part two": "9:30 PM only",
      "oppenheimer": "Sold out for tonight",
  }
  return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

#### customize tool name and description

In [10]:
@tool('book_seats', description = 'Book Cinema for a customer, use whenever customer wants to book/reserve a seat.')
def reserve(movie:str,seats:int) ->str:
  """Reserve Seats"""
  return f"Reserved {seats} seat for {movie}"

### langchain builtin tools

#### need tavily api key for this

In [6]:
from langchain_tavily import TavilySearch

In [11]:
@tool('search_internet_with_tavily', description = 'Use this when user wants to search the internet with Tavily')
def search_internet(topic):
  return TavilySearch()

try:
    tool = TavilySearch(
        max_results=5,
        topic="general",
        # include_answer=False,
        # include_raw_content=False,
        # include_images=False,
        # include_image_descriptions=False,
        # search_depth="basic",
        # time_range="day",
        # include_domains=None,
        # exclude_domains=None
    )
except Exception as e:
    print(e)

1 validation error for TavilySearchAPIWrapper
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


### arguments schema

In [12]:
from pydantic import BaseModel,Field
from typing import Literal

In [13]:
class SeatBookingInput(BaseModel):
    movie_title:str = Field(description='Exact Movie Title')
    seat_count : int = Field(description='Number of seats to book', ge=1, le=10)
    preferred_row : Literal['front', 'middle', 'back'] = Field(default='middle', description='Preferred seat row')

In [14]:
@tool
def book_seats(movie_title:str, seat_count:int, preferred_row:str)-> str:
  """Book Seats for a Movie"""
  return f"Booked {seats} seats for {movie_title} in row {preferred_row}"

In [16]:
def book_seats1(movie_title:str, seat_count:int, preferred_row:str)-> str:
  """Book Seats for a Movie"""
  return f"Booked {seats} seats for {movie_title} in row {preferred_row}"

In [15]:
book_seats.args

{'movie_title': {'title': 'Movie Title', 'type': 'string'},
 'seat_count': {'title': 'Seat Count', 'type': 'integer'},
 'preferred_row': {'title': 'Preferred Row', 'type': 'string'}}

In [17]:
book_seats1.args  ##regular methods and does not have any argument schema

AttributeError: 'function' object has no attribute 'args'

### config and runtime cannot be used in tool arguments

In [21]:
from pydantic import BaseModel,Field
from typing import Literal
from langchain.agents import create_agent

In [26]:
class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

@tool
def get_weather(location: str,config:str,units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} {config} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [27]:
agent = create_agent(
    model='openai:gpt-5-mini',
    tools=[get_weather]
    )

In [28]:
try:
    response = agent.invoke(
            {"messages": [{"role": "user", "content": "What is the weather in Delhi in celsius and tell the forecast?"}]},
    )
except Exception as e:
    print(e)

get_weather() missing 1 required positional argument: 'config'


In [32]:
@tool
def get_weather1(location: str,runtime:str,units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} {runtime} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result
    
agent = create_agent(
    model='openai:gpt-5-mini',
    tools=[get_weather1]
    )
try:
    response = agent.invoke(
            {"messages": [{"role": "user", "content": "What is the weather in Delhi in celsius and tell the forecast?"}]},
    )
    print(response)
except Exception as e:
    print(e)

{'messages': [HumanMessage(content='What is the weather in Delhi in celsius and tell the forecast?', additional_kwargs={}, response_metadata={}, id='01faf78e-aaa4-4cce-a46f-490824dff737'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 170, 'prompt_tokens': 167, 'total_tokens': 337, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E6Ti4AMYT2CUiL7atGcmWd4Rvxs4G', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fa6fa-b656-7bb1-a656-b4f7ead350b5-0', tool_calls=[{'name': 'get_weather1', 'args': {'location': 'New Delhi, India', 'runtime': 'current', 'units': 'celsius', 'include_forecast':

### tool binding

In [6]:
@tool
def check_showtimes(movie_title:str) -> str:
  """Check available showtimes for a movie at the cinema."""
  return "Show is available"

@tool
def book_seats(movie_title:str, seat_count:int, preferred_row:str)-> str:
  """Book Seats for a Movie"""
  return f"Booked {seats} seats for {movie_title} in row {preferred_row}"

In [7]:
model_with_tools = model.bind_tools([check_showtimes,book_seats])

In [8]:
response = model_with_tools.invoke("Is Interstellar show available tonight?")

In [9]:
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 170, 'total_tokens': 196, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E7Ajh5B1FWihiMtllhvAGwJJl5hPJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb0d6-709a-7f52-b1cf-12bd1854cdda-0', tool_calls=[{'name': 'check_showtimes', 'args': {'movie_title': 'Interstellar'}, 'id': 'call_FNNMiJFEnFiVDQ7IoIBXZeDU', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 26, 'total_tokens': 196, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio'

### Runtime parameter in Tools

In [10]:
from langchain.tools import tool,ToolRuntime
from langchain_core.messages import HumanMessage

@tool
def get_last_movie_mentioned(movie:str, runtime:ToolRuntime) -> str:
  """Get the last movie mentioned in the chat history."""
  pass

print(get_last_movie_mentioned.args)

{'movie': {'title': 'Movie', 'type': 'string'}}


In [11]:
from langgraph.store.memory import InMemoryStore
from langchain_core.tools import tool
from langchain.tools import tool,ToolRuntime
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

In [17]:
from typing import Any


loyalty_store= InMemoryStore()


@tool
def save_favourite_genres(customer_id:str,genre:str,runtime:ToolRuntime) -> str:
    """Save a customer's facvourite movie genre for future visits"""
    print('inside save_favourite_genres customer_id '+customer_id)
    print(runtime)
    runtime.store.put((customer_id,"preferences"),"favourite_genre",{"value":genre})
    return f"Got it -- I will remmeber you like {genre} movies"

@tool
def recall_favourite_genre(customer_id:str,runtime:ToolRuntime) -> str:
    """ Recall a customer's fav movie genre, if we have saved it before"""
    print('inside recall_favourite_genre customer_id '+customer_id)
    print(runtime)
    favourite_genre = runtime.store.get((customer_id,"preferences"),"favourite_genre")
    return favourite_genre.value["value"] if favourite_genre else "We don't have any saved preference for this user"


memory_agent = create_agent(
    model = model,
    tools=[save_favourite_genres,recall_favourite_genre],
    store=loyalty_store  # Attached to the agent, tools can access it using runtime
)

In [18]:
memory_agent.invoke({"messages": [("user", "Hi, I'm customer priya_01, I love sci-fi movies, please remember that.")]})

inside save_favourite_genres customer_id priya_01
ToolRuntime(state={'messages': [HumanMessage(content="Hi, I'm customer priya_01, I love sci-fi movies, please remember that.", additional_kwargs={}, response_metadata={}, id='16addc05-c12d-4ea3-8131-31fec15d1986'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 190, 'total_tokens': 290, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E7At6kEdmUjMLi3ZAmM38YyGt9A3D', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb0df-59dd-77a0-a454-96e1b1624be2-0', tool_calls=[{'name': 'save_favourite_genres', 'args': {'customer

{'messages': [HumanMessage(content="Hi, I'm customer priya_01, I love sci-fi movies, please remember that.", additional_kwargs={}, response_metadata={}, id='16addc05-c12d-4ea3-8131-31fec15d1986'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 190, 'total_tokens': 290, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E7At6kEdmUjMLi3ZAmM38YyGt9A3D', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb0df-59dd-77a0-a454-96e1b1624be2-0', tool_calls=[{'name': 'save_favourite_genres', 'args': {'customer_id': 'priya_01', 'genre': 'sci-fi'}, 'id': 'call_hBAse3jxG8xI9yEk

In [19]:
memory_agent.invoke({"messages": [("user", "Hi, I'm customer priya_01, what kind of movie do I like ?")]})

inside recall_favourite_genre customer_id priya_01
ToolRuntime(state={'messages': [HumanMessage(content="Hi, I'm customer priya_01, what kind of movie do I like ?", additional_kwargs={}, response_metadata={}, id='12d39cf0-fb80-4614-88e0-76179a201c63'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 188, 'total_tokens': 283, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E7AvGgaQ4wYSD3EfhTLTEYDUFVpfB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb0e1-660d-77a3-8d81-6f1044e5f7ed-0', tool_calls=[{'name': 'recall_favourite_genre', 'args': {'customer_id': 'priya

{'messages': [HumanMessage(content="Hi, I'm customer priya_01, what kind of movie do I like ?", additional_kwargs={}, response_metadata={}, id='12d39cf0-fb80-4614-88e0-76179a201c63'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 188, 'total_tokens': 283, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E7AvGgaQ4wYSD3EfhTLTEYDUFVpfB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb0e1-660d-77a3-8d81-6f1044e5f7ed-0', tool_calls=[{'name': 'recall_favourite_genre', 'args': {'customer_id': 'priya_01'}, 'id': 'call_lS51zQgn2aW5OBvujYDBC0Ly', 'type': 'tool_call'}]

In [20]:
items = loyalty_store.search(("priya_01", "preferences"))

In [21]:
for item in items:
    print(item)

Item(namespace=['priya_01', 'preferences'], key='favourite_genre', value={'value': 'sci-fi'}, created_at='2026-07-30T02:34:17.926058+00:00', updated_at='2026-07-30T02:34:17.926063+00:00', score=None)


### Return tool calling result directly to user dont call LLM again to create AIMessage

In [25]:
from langchain.tools import tool

@tool(return_direct=True)
def get_weather(city: str):
    """return the weather"""
    return f"It's 75°F in {city}."

agent = create_agent(
    model = model,
    tools=[get_weather],
    store=loyalty_store  # Attached to the agent, tools can access it using runtime
)

response = agent.invoke({"messages": [("user", "what is the weatheweather in chicago? ")]})
print(response)

{'messages': [HumanMessage(content='what is the weathe in chicago? ', additional_kwargs={}, response_metadata={}, id='de0ed76e-5248-48c8-abb1-7dbfcdb10e64'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 131, 'total_tokens': 154, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E7B1uFt8V45aaQbFvQ0qo1hfLdnuY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb0e7-b154-7b80-a22b-b6dac9e56969-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Chicago'}, 'id': 'call_390vBnLcxMLydecALRdtgr6a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

### Dynamic Tool Loading & Calling